In [ ]:
# Path configuration — edit these to match your local setup, then run the rest of the notebook.
# (Everything below in this notebook reads from these variables instead of hardcoded paths.)

# Root of the preprocessed LibriTTS-R data (see README "Preprocessed data layout"),
# i.e. the directory containing train-clean-100-preprocessed/, dev-clean-preprocessed/,
# test-clean-preprocessed/, and the {split}.json manifests.
LIBRITTS_ROOT = "/path/to/LibriTTS_R/"

# Root that conf/train/*.yaml's checkpoint.dirpath / logger.save_dir point at
# (same value you passed as train.checkpoint.dirpath / train.logger.save_dir when training).
TRAIN_OUTPUT_ROOT = "/path/to/output"


LibriTTS

In [16]:
import tts
import torch
import hydra
from hydra.core.global_hydra import GlobalHydra
import lightning as L
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.stats import pearsonr

GlobalHydra.instance().clear()
L.seed_everything(42, workers=True)
torch.set_float32_matmul_precision('medium')
with hydra.initialize(version_base=None, config_path="conf"):
    config = hydra.compose(config_name="config", overrides=["model=large_model","train=train_large"])

Seed set to 42


In [ ]:
config['train']['logger']['save_dir'] = config['train']['logger']['save_dir'].format(experiment_name=config['train']['experiment_name'])
config['train']['checkpoint']['dirpath'] = config['train']['checkpoint']['dirpath'].format(experiment_name=config['train']['experiment_name'])
config['train']['trainer']['devices'] = 1
config['train']['trainer']['strategy'] = 'auto'
trainer = L.Trainer(**config['train']['trainer'],
        callbacks=[ModelCheckpoint(**config['train']['checkpoint']),],
        logger=TensorBoardLogger(**config['train']['logger']),
        limit_predict_batches=1)
config['train']['datamodule']['batch_size'] = 8
config['train']['datamodule']['num_workers'] = 4
datamodule = tts.LibriTTSDataModule(config)
model = tts.LitTTS(config)
config['model']['tts']['use_aligner_durations_if_possible'] = True
model_align = tts.LitTTS(config)
CKPT_PATH = f'{TRAIN_OUTPUT_ROOT}/ddp_slurm_large_model/ckpt/step=32000-val/loss=6.03.ckpt'  # paper checkpoint
output = trainer.predict(model, datamodule, ckpt_path=CKPT_PATH, weights_only=False)
output_align = trainer.predict(model_align, datamodule, ckpt_path=CKPT_PATH, weights_only=False)

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os

def plot_sparc(prediction, ground_truth=None, aligned=None):
    # Create a figure with 14 subplots stacked vertically
    fig, axes = plt.subplots(14, 1, figsize=(14, 15), sharex=True)
    
    names = ["ULX", "ULY", "LLX", "LLY", "LIX", "LIY", "TTX", "TTY", "TBX", "TBY", "TDX", "TDY", "Pitch", "Loudness"]
    for ax in axes[:-1]:
        ax.label_outer()
    for i in range(14):
        for spine in axes[i].spines.values():
            spine.set_visible(False)
        # total_dur = 0
        axes[i].set_yticks([]) 
        axes[i].set_xticks([]) 
        # axes[i].set_ylim(-3, 3)
        # axes[i].set_xlim(0, 100)
        line1, = axes[i].plot(prediction[:, i], color='green', linewidth=6)
        if i == 0:
            line1.set_label(f'STArK')
        if aligned is not None:
            line2, = axes[i].plot(aligned[:, i], color='orange', linestyle='--', linewidth=6)
            if i == 0:
                line2.set_label(f'STArK+align')
        if ground_truth is not None:
            line3, = axes[i].plot(ground_truth[:, i], color='purple', linestyle=':', linewidth=6)
            if i == 0:
                line3.set_label(f'SPARC')

        axes[i].set_ylabel(f'{names[i]}', fontsize=38, rotation=0, labelpad=0, ha='right', va='center')

    # Set a general label for the x-axis
    plt.xlabel('')
    fig.legend(loc='upper right', fontsize=32)

    # Adjust layout for better visualization
    plt.tight_layout(pad=0)
    # plt.tight_layout(pad=3.0)  # Add more padding between subplots
    # plt.subplots_adjust(top=0.95, bottom=0.05, hspace=0.1)  # More space at the top/bottom


    # Show the plots
    plt.show()

    # corr = {item[0]: float(item[1]) for item in zip(names, list(pearsonr(prediction, ground_truth).statistic))}
    # return corr, sum(corr.values()) / len(corr)

In [ ]:
for i in range(8):
    plot_sparc(output[0][1][0][i][:(~output[0][1][1][i]).sum(),:].cpu().numpy(), 
               np.load(os.path.join(LIBRITTS_ROOT, "test-clean-preprocessed", "emasrc", f"{output[0][0][i]}.ema.npy",)),
               output_align[0][1][0][i][:(~output_align[0][1][1][i]).sum(),:].cpu().numpy())


PCC and DTW

In [1]:
import tts
import torch
import hydra
from hydra.core.global_hydra import GlobalHydra
import lightning as L
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint

GlobalHydra.instance().clear()
L.seed_everything(42, workers=True)
torch.set_float32_matmul_precision('medium')
with hydra.initialize(version_base=None, config_path="conf"):
    config = hydra.compose(config_name="config", overrides=["model=large_model","train=train_large"])

Seed set to 42


In [ ]:
config['train']['logger']['save_dir'] = config['train']['logger']['save_dir'].format(experiment_name=config['train']['experiment_name'])
config['train']['checkpoint']['dirpath'] = config['train']['checkpoint']['dirpath'].format(experiment_name=config['train']['experiment_name'])
config['train']['trainer']['devices'] = 1
config['train']['trainer']['strategy'] = 'auto'
trainer = L.Trainer(**config['train']['trainer'],
        callbacks=[ModelCheckpoint(**config['train']['checkpoint']),],
        logger=TensorBoardLogger(**config['train']['logger']),)
config['train']['datamodule']['batch_size'] = 16
config['train']['datamodule']['num_workers'] = 16
config['model']['tts']['use_aligner_durations_if_possible'] = False
datamodule = tts.LibriTTSDataModule(config)
model = tts.LitTTS(config)
config['model']['tts']['use_aligner_durations_if_possible'] = True
model_align = tts.LitTTS(config)
CKPT_PATH = f'{TRAIN_OUTPUT_ROOT}/ddp_slurm_large_model/ckpt/step=32000-val/loss=6.03.ckpt'  # paper checkpoint
output = trainer.predict(model, datamodule, ckpt_path=CKPT_PATH, weights_only=False)
output_align = trainer.predict(model_align, datamodule, ckpt_path=CKPT_PATH, weights_only=False)

In [9]:
print(len(output), len(output_align))

302 302


In [10]:
from scipy.stats import pearsonr
from tslearn.metrics import dtw
import numpy as np
import os
from tqdm.notebook import tqdm
def compute_pearson(prediction, ground_truth):
    names = ["ULX", "ULY", "LLX", "LLY", "LIX", "LIY", "TTX", "TTY", "TBX", "TBY", "TDX", "TDY", "Pitch", "Loudness"]
    corr = {item[0]: float(item[1]) for item in zip(names, list(pearsonr(prediction, ground_truth, axis=0).statistic))}
    # return mean corr of first 12 features without pitch and loudness, corr of pitch, corr of loudness
    return sum([v for k, v in corr.items() if k not in ['Pitch', 'Loudness']]) / (len(corr) - 2), corr['Pitch'], corr['Loudness']
def compute_dtw(prediction, ground_truth):
    names = ["ULX", "ULY", "LLX", "LLY", "LIX", "LIY", "TTX", "TTY", "TBX", "TBY", "TDX", "TDY", "Pitch", "Loudness"]
    dtw_distances = {item[0]: float(item[1]) for item in zip(names, [dtw(prediction[:, i], ground_truth[:, i]) for i in range(prediction.shape[1])])}
    # return mean dtw of first 12 features without pitch and loudness, dtw of pitch, dtw of loudness
    return sum([v for k, v in dtw_distances.items() if k not in ['Pitch', 'Loudness']]) / (len(dtw_distances) - 2), dtw_distances['Pitch'], dtw_distances['Loudness']


In [18]:
output[0][1][0][0].shape

torch.Size([891, 14])

In [ ]:
dtw_ema_stark = []
dtw_pitch_stark = []
dtw_loudness_stark = []
for batch in tqdm(range(len(output))):
    for i in range(len(output[batch][0])):
        dtw_ema, dtw_pitch, dtw_loudness = compute_dtw(
            output[batch][1][0][i][:(~output[batch][1][1][i]).sum(),:].cpu().numpy(), 
            np.load(os.path.join(LIBRITTS_ROOT, "test-clean-preprocessed", "emasrc", f"{output[batch][0][i]}.ema.npy",)))
        dtw_ema_stark.append(dtw_ema)
        dtw_pitch_stark.append(dtw_pitch)
        dtw_loudness_stark.append(dtw_loudness)

In [12]:
print(f"DTW EMA: {np.mean(dtw_ema_stark):.3f}$\\pm${1.96 * np.std(dtw_ema_stark) / np.sqrt(len(dtw_ema_stark)):.3f}")
print(f"DTW Pitch: {np.mean(dtw_pitch_stark):.3f}$\\pm${1.96 * np.std(dtw_pitch_stark) / np.sqrt(len(dtw_pitch_stark)):.3f}")
print(f"DTW Loudness: {np.mean(dtw_loudness_stark):.3f}$\\pm${1.96 * np.std(dtw_loudness_stark) / np.sqrt(len(dtw_loudness_stark)):.3f}")

DTW EMA: 5.142$\pm$0.065
DTW Pitch: 1.625$\pm$0.027
DTW Loudness: 3.572$\pm$0.046


In [ ]:
pearson_ema_stark_align, dtw_ema_stark_align = [], []
pearson_pitch_stark_align, dtw_pitch_stark_align = [], []
pearson_loudness_stark_align, dtw_loudness_stark_align = [], []
for batch in tqdm(range(len(output_align))):
    for i in range(len(output_align[batch][0])):
        pearson_ema, pearson_pitch, pearson_loudness = compute_pearson(
            output_align[batch][1][0][i][:(~output_align[batch][1][1][i]).sum(),:].cpu().numpy(), 
            np.load(os.path.join(LIBRITTS_ROOT, "test-clean-preprocessed", "emasrc", f"{output_align[batch][0][i]}.ema.npy",)))
        dtw_ema, dtw_pitch, dtw_loudness = compute_dtw(
            output_align[batch][1][0][i][:(~output_align[batch][1][1][i]).sum(),:].cpu().numpy(), 
            np.load(os.path.join(LIBRITTS_ROOT, "test-clean-preprocessed", "emasrc", f"{output_align[batch][0][i]}.ema.npy",)))
        pearson_ema_stark_align.append(pearson_ema)
        dtw_ema_stark_align.append(dtw_ema)
        pearson_pitch_stark_align.append(pearson_pitch)
        dtw_pitch_stark_align.append(dtw_pitch)
        pearson_loudness_stark_align.append(pearson_loudness)
        dtw_loudness_stark_align.append(dtw_loudness)

In [77]:
all(np.array(dtw_no_pitch_stark_align) > 0)

True

In [14]:
# print mean and 95%CI for pearson and dtw with and without pitch
print(f"Pearson EMA: {np.mean(pearson_ema_stark_align):.3f}$\\pm${1.96 * np.std(pearson_ema_stark_align) / np.sqrt(len(pearson_ema_stark_align)):.3f}")
print(f"Pearson Pitch: {np.mean(pearson_pitch_stark_align):.3f}$\\pm${1.96 * np.std(pearson_pitch_stark_align) / np.sqrt(len(pearson_pitch_stark_align)):.3f}")
print(f"Pearson Loudness: {np.mean(pearson_loudness_stark_align):.3f}$\\pm${1.96 * np.std(pearson_loudness_stark_align) / np.sqrt(len(pearson_loudness_stark_align)):.3f}")
print(f"DTW EMA: {np.mean(dtw_ema_stark_align):.3f}$\\pm${1.96 * np.std(dtw_ema_stark_align) / np.sqrt(len(dtw_ema_stark_align)):.3f}")
print(f"DTW Pitch: {np.mean(dtw_pitch_stark_align):.3f}$\\pm${1.96 * np.std(dtw_pitch_stark_align) / np.sqrt(len(dtw_pitch_stark_align)):.3f}")
print(f"DTW Loudness: {np.mean(dtw_loudness_stark_align):.3f}$\\pm${1.96 * np.std(dtw_loudness_stark_align) / np.sqrt(len(dtw_loudness_stark_align)):.3f}")

Pearson EMA: 0.905$\pm$0.001
Pearson Pitch: 0.533$\pm$0.009
Pearson Loudness: 0.796$\pm$0.002
DTW EMA: 4.330$\pm$0.055
DTW Pitch: 1.581$\pm$0.027
DTW Loudness: 3.372$\pm$0.044


In [15]:
# print difference between stark and stark_align for dtw with and without pitch
# print difference of means and sum of 95%CI for dtw with and without pitch
dtw_diff_ema = np.mean(dtw_ema_stark_align) - np.mean(dtw_ema_stark)
dtw_diff_pitch = np.mean(dtw_pitch_stark_align) - np.mean(dtw_pitch_stark)
dtw_diff_loudness = np.mean(dtw_loudness_stark_align) - np.mean(dtw_loudness_stark)
dtw_diff_ema_ci = 1.96 * np.sqrt(np.var(dtw_ema_stark_align) / len(dtw_ema_stark_align) + np.var(dtw_ema_stark) / len(dtw_ema_stark))
dtw_diff_pitch_ci = 1.96 * np.sqrt(np.var(dtw_pitch_stark_align) / len(dtw_pitch_stark_align) + np.var(dtw_pitch_stark) / len(dtw_pitch_stark))
dtw_diff_loudness_ci = 1.96 * np.sqrt(np.var(dtw_loudness_stark_align) / len(dtw_loudness_stark_align) + np.var(dtw_loudness_stark) / len(dtw_loudness_stark))
print(f"DTW difference EMA: {dtw_diff_ema:.2f}$\\pm${dtw_diff_ema_ci:.2f}")
print(f"DTW difference Pitch: {dtw_diff_pitch:.2f}$\\pm${dtw_diff_pitch_ci:.2f}")
print(f"DTW difference Loudness: {dtw_diff_loudness:.2f}$\\pm${dtw_diff_loudness_ci:.2f}")

DTW difference EMA: -0.81$\pm$0.09
DTW difference Pitch: -0.04$\pm$0.04
DTW difference Loudness: -0.20$\pm$0.06


In [9]:
from tts.sparc import load_model
import json
import os
import numpy as np
from IPython.display import Audio, display
import soundfile as sf

In [10]:
sparc_model = load_model(model_name='en+', config="conf/sparc/model_englishplus_2M.yaml")

/home/YOUR_USERNAME/articulatory-tts/.venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [ ]:
for i in range(8):
    pitch_stats = json.load(open(os.path.join(LIBRITTS_ROOT, "test-clean-preprocessed", "pitch_stats.json"), "r"))
    median_pitch = pitch_stats[output[0][0][i]]
    spk_emb = np.load(os.path.join(LIBRITTS_ROOT, "test-clean-preprocessed", "spk_emb", f"{output[0][0][i]}.npy"))
    ema = output[0][1][0][i][:(~output[0][1][1][i]).sum(),:].cpu().numpy()
    ema2 = np.copy(ema)
    ema2[:,12] = np.multiply(np.exp(ema[:,12]), median_pitch)
    wav1 = sparc_model.decode(ema2[:, :12], ema2[:,12], ema2[:,13], spk_emb)
    sf.write(f'samples/test_synth_{output[0][0][i]}.wav', wav1, sparc_model.sr)

    ema = np.load(os.path.join(LIBRITTS_ROOT, "test-clean-preprocessed", "emasrc", f"{output[0][0][i]}.ema.npy"))
    ema[:,12] = np.multiply(np.exp(ema[:,12]), median_pitch)
    wav2 = sparc_model.decode(ema[:, :12], ema[:,12], ema[:,13], spk_emb)
    print(f"{output[0][0][i]}: 1. Synthesized, 2. Ground truth")
    sf.write(f'samples/test_gt_{output[0][0][i]}.wav', wav2, sparc_model.sr)
    display(Audio(wav1, rate=16000), Audio(wav2, rate=16000))

In [3]:
import tts
import torch
import hydra
from hydra.core.global_hydra import GlobalHydra
import lightning as L
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm
import os
from scipy.stats import pearsonr
import json

GlobalHydra.instance().clear()
L.seed_everything(42, workers=True)
torch.set_float32_matmul_precision('medium')
with hydra.initialize(version_base=None, config_path="conf"):
    config = hydra.compose(config_name="config", overrides=["model=large_model","train=train_large"])

In [ ]:
config['train']['logger']['save_dir'] = config['train']['logger']['save_dir'].format(experiment_name=config['train']['experiment_name'])
config['train']['checkpoint']['dirpath'] = config['train']['checkpoint']['dirpath'].format(experiment_name=config['train']['experiment_name'])
config['train']['trainer']['devices'] = 1
config['train']['trainer']['strategy'] = 'auto'
trainer = L.Trainer(**config['train']['trainer'],
        callbacks=[ModelCheckpoint(**config['train']['checkpoint']),],
        logger=TensorBoardLogger(**config['train']['logger']),)
config['train']['datamodule']['batch_size'] = 8
config['train']['datamodule']['num_workers'] = 4
config['model']['tts']['use_aligner_durations_if_possible'] = True
datamodule = tts.LibriTTSDataModule(config)
model = tts.LitTTS(config)
CKPT_PATH = f'{TRAIN_OUTPUT_ROOT}/ddp_slurm_large_model/ckpt/step=32000-val/loss=6.03.ckpt'  # paper checkpoint
output = trainer.predict(model, datamodule, ckpt_path=CKPT_PATH, weights_only=False)
len(output)

In [ ]:
output_dir = f"{TRAIN_OUTPUT_ROOT}/libritts_r_test-clean_pred_large_32000-gt-prosody"
os.makedirs(os.path.join(output_dir, "ema"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "wav"), exist_ok=True)
for batch in tqdm(output):
    for id, sparc, mask in zip(batch[0], batch[1][0], batch[1][1]):
        print(id, len(sparc), len(mask))
        print(sparc.shape)
        sparc = sparc[:(~mask).sum(),:].cpu().numpy()
        print(sparc.shape)
        # np.save(os.path.join(output_dir, "ema", f"{id}.pred.npy"), sparc)
        
        pitch_stats = json.load(open(os.path.join(LIBRITTS_ROOT, "test-clean-preprocessed", "pitch_stats.json"), "r"))
        median_pitch = pitch_stats[id]
        spk_emb = np.load(os.path.join(LIBRITTS_ROOT, "test-clean-preprocessed", "spk_emb", f"{id}.npy"))
        gt_ema = np.load(os.path.join(LIBRITTS_ROOT, "test-clean-preprocessed", "emasrc", f"{id}.ema.npy")) # gt prosody
        ema2 = np.copy(sparc)

        # ema2[:,12] = np.multiply(np.exp(sparc[:,12]), median_pitch)

        ema2[:,12:] = gt_ema[:,12:] # gt prosody
        np.save(os.path.join(output_dir, "ema", f"{id}.pred.npy"), ema2) # gt prosody
        ema2[:,12] = np.multiply(np.exp(ema2[:,12]), median_pitch) # gt prosody
        wav1 = sparc_model.decode(ema2[:, :12], ema2[:,12], ema2[:,13], spk_emb)
        sf.write(os.path.join(output_dir, "wav", f"{id}.pred.wav"), wav1, sparc_model.sr)

In [8]:
from torchmetrics.functional.audio.dnsmos import deep_noise_suppression_mean_opinion_score as dnsmos
from tqdm.notebook import tqdm
import os
import soundfile as sf
import librosa
import numpy as np
import torch

In [ ]:
# output_dir = f"{TRAIN_OUTPUT_ROOT}/libritts_r_test-clean_pred_large_32000"
output_dir = f"{TRAIN_OUTPUT_ROOT}/libritts_r_test-clean_pred_large_32000-gt-align"
# output_dir = f"{TRAIN_OUTPUT_ROOT}/libritts_r_test-clean_pred_large_32000-gt-prosody"

dnsmos_scores = {"p808": [], "sig": [], "bak": [], "ovr": [], }
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
for file in tqdm(os.listdir(os.path.join(output_dir, "wav"))):
    if file.endswith(".pred.wav"):
        id = file.split(".pred.wav")[0]
        pred_wav, sr = sf.read(os.path.join(output_dir, "wav", f"{id}.pred.wav"))
        score = dnsmos(torch.from_numpy(pred_wav).to(device), sr, personalized=False, num_threads=8)
        p808, sig, bak, ovr = score
        dnsmos_scores["p808"].append(p808)
        dnsmos_scores["sig"].append(sig)
        dnsmos_scores["bak"].append(bak)
        dnsmos_scores["ovr"].append(ovr)
print(f"Average p808 score: {np.mean(dnsmos_scores['p808']):.3f}, 95% CI: {1.96 * np.std(dnsmos_scores['p808']) / np.sqrt(len(dnsmos_scores['p808'])):.3f}")
print(f"Average sig score: {np.mean(dnsmos_scores['sig']):.3f}, 95% CI: {1.96 * np.std(dnsmos_scores['sig']) / np.sqrt(len(dnsmos_scores['sig'])):.3f}")
print(f"Average bak score: {np.mean(dnsmos_scores['bak']):.3f}, 95% CI: {1.96 * np.std(dnsmos_scores['bak']) / np.sqrt(len(dnsmos_scores['bak'])):.3f}")
print(f"Average ovr score: {np.mean(dnsmos_scores['ovr']):.3f}, 95% CI: {1.96 * np.std(dnsmos_scores['ovr']) / np.sqrt(len(dnsmos_scores['ovr'])):.3f}")

In [ ]:
# output_dir = os.path.join(LIBRITTS_ROOT, "test-clean-preprocessed/wav")
# output_dir = os.path.join(LIBRITTS_ROOT, "test-clean-sparc-resynth")
# output_dir = os.path.join(LIBRITTS_ROOT, "test-clean-your_tts-synthesized")
output_dir = os.path.join(LIBRITTS_ROOT, "test-clean-fastpitch-synthesized")

dnsmos_scores = {"p808": [], "sig": [], "bak": [], "ovr": [], }
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
for file in tqdm(os.listdir(output_dir)):
    if file.endswith(".wav"):
        id = file.split(".wav")[0]
        pred_wav, sr = sf.read(os.path.join(output_dir, f"{id}.wav"))
        assert sr != 16000, f"Expected sample rate of 16000, but got {sr}"
        pred_wav = librosa.resample(pred_wav, orig_sr=sr, target_sr=16000)
        score = dnsmos(torch.from_numpy(pred_wav).to(device), 16000, personalized=False, num_threads=8)
        p808, sig, bak, ovr = score
        dnsmos_scores["p808"].append(p808)
        dnsmos_scores["sig"].append(sig)
        dnsmos_scores["bak"].append(bak)
        dnsmos_scores["ovr"].append(ovr)
print(f"Average p808 score: {np.mean(dnsmos_scores['p808']):.3f}, 95% CI: {1.96 * np.std(dnsmos_scores['p808']) / np.sqrt(len(dnsmos_scores['p808'])):.3f}")
print(f"Average sig score: {np.mean(dnsmos_scores['sig']):.3f}, 95% CI: {1.96 * np.std(dnsmos_scores['sig']) / np.sqrt(len(dnsmos_scores['sig'])):.3f}")
print(f"Average bak score: {np.mean(dnsmos_scores['bak']):.3f}, 95% CI: {1.96 * np.std(dnsmos_scores['bak']) / np.sqrt(len(dnsmos_scores['bak'])):.3f}")
print(f"Average ovr score: {np.mean(dnsmos_scores['ovr']):.3f}, 95% CI: {1.96 * np.std(dnsmos_scores['ovr']) / np.sqrt(len(dnsmos_scores['ovr'])):.3f}")

In [17]:
torch.cuda.empty_cache()

Generate MOS data

In [1]:
import json
import os
import random
import soundfile as sf
random.seed(42)


In [ ]:
with open(os.path.join(LIBRITTS_ROOT, "test-clean.json"), "r") as f:
    ids = json.load(f)
samples = random.sample(ids, 24)
samples

In [ ]:
wav_dirs = [("art-tts", f"{TRAIN_OUTPUT_ROOT}/libritts_r_test-clean_pred_large_32000/wav"),
            ("art-tts-align", f"{TRAIN_OUTPUT_ROOT}/libritts_r_test-clean_pred_large_32000-gt-align/wav"),
            ("art-tts-prosody", f"{TRAIN_OUTPUT_ROOT}/libritts_r_test-clean_pred_large_32000-gt-prosody/wav"),
            ("gt", os.path.join(LIBRITTS_ROOT, "test-clean-preprocessed/wav")),
            ("sparc-resynth", os.path.join(LIBRITTS_ROOT, "test-clean-sparc-resynth")),
            ("your-tts", os.path.join(LIBRITTS_ROOT, "test-clean-your_tts-synthesized"))]


In [11]:
j = 0
for i, sample in enumerate(samples):
    rotated = wav_dirs[-j:] + wav_dirs[:-j]
    for k, v in rotated:
        if "art-tts" in k:
            wav_path = os.path.join(v, f"{sample}.pred.wav")
        else:
            wav_path = os.path.join(v, f"{sample}.wav")
        # copy wav file to mos_eval_data
        wav_file = sf.read(wav_path)
        sf.write(os.path.join("mos_eval_data", f"mos_eval_set_{j}", f"{k}_{sample}.wav"), wav_file[0], wav_file[1])
    print(f"Sample {i+1}: {sample}")
    j += 1
    j %= len(wav_dirs)

Sample 1: 1580_141083_000057_000000
Sample 2: 5142_33396_000000_000001
Sample 3: 1284_1180_000050_000001
Sample 4: 237_126133_000009_000000
Sample 5: 260_123286_000048_000000
Sample 6: 8463_287645_000030_000001
Sample 7: 7729_102255_000011_000003
Sample 8: 4077_13754_000005_000001
Sample 9: 5683_32865_000045_000003
Sample 10: 5683_32866_000025_000000
Sample 11: 4446_2275_000033_000001
Sample 12: 4446_2273_000035_000000
Sample 13: 3570_5695_000002_000000
Sample 14: 260_123288_000006_000000
Sample 15: 8463_294828_000047_000001
Sample 16: 4446_2273_000001_000008
Sample 17: 1995_1826_000033_000000
Sample 18: 237_134493_000007_000000
Sample 19: 4446_2275_000043_000006
Sample 20: 7729_102255_000013_000001
Sample 21: 5639_40744_000010_000005
Sample 22: 8455_210777_000022_000010
Sample 23: 8555_284449_000042_000000
Sample 24: 2300_131720_000030_000001


In [14]:
import json

for i in range(6):
    json.dump(os.listdir(os.path.join('mos_eval_data', f'mos_eval_set_{i}')), open(f'mos_eval_data/mos_eval_set_{i}/manifest.json', 'w'), indent=4)

In [ ]:
pitch_stats = json.load(open(os.path.join(LIBRITTS_ROOT, "test-clean-preprocessed", "pitch_stats.json"), "r"))
print(pitch_stats)

In [ ]:
ema = np.load(os.path.join(output_dir, 'cleft_21M_Tiny_Tim.npy'))
spk_emb = np.load(os.path.join(LIBRITTS_ROOT, "test-clean-preprocessed", "spk_emb", f"2300_131720_000038_000001.npy"))
ema[:,12] = np.multiply(np.exp(ema[:,12]), pitch_stats['2300_131720_000038_000001'])
wav2 = sparc_model.decode(ema[:, :12], ema[:,12], ema[:,13], spk_emb)
sf.write(f'samples/test_star_cleft_21M_Tiny_Tim_2300_131720_000038_000001.wav', wav2, sparc_model.sr)
display(Audio(wav2, rate=16000))